# Bonus 2 — Guardrails for Production RAG

> **Duration:** ~45 minutes  
> **Prerequisite:** Notebooks 04 and 05 (pipeline + evaluation)

A RAG system that retrieves the right chunks and generates a grounded answer is a great start.
In production — especially in regulated sectors like insurance — you also need **guardrails**:
automated checks that prevent the pipeline from producing harmful or misleading outputs.

```
User question
     ↓
[1] Scope filter  ──►  off-topic?  ►  blocked (“This assistant handles insurance only”)
     ↓ on-topic
[2] Retrieval + confidence check  ──►  low similarity?  ►  blocked (“No relevant info found”)
     ↓ confident
[3] Generate answer
     ↓
[4] Faithfulness gate  ──►  not grounded?  ►  flagged for review
     ↓ grounded
Final answer ✔
```

**What you will build:**

| # | Guardrail | Technique |
|---|-----------|-----------|
| 1 | Input scope filter | LLM classifier |
| 2 | Retrieval confidence check | Similarity threshold |
| 3 | Faithfulness gate | LLM verifier |
| 4 | `GuardedRAGPipeline` | Combines all three |

In [ ]:
import os
import sys
import json
import warnings
warnings.filterwarnings("ignore")

sys.path.insert(0, "..")

import chromadb
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

from src.vector_store import retrieve

load_dotenv("../.env")

# --- Embedding model & ChromaDB ---
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

chroma_client = chromadb.PersistentClient(path="../chroma_db")
collection = chroma_client.get_or_create_collection(
    name="workshop_rag",
    metadata={"hnsw:space": "cosine"},
)
print(f"Collection: {collection.count()} docs")
if collection.count() == 0:
    print("Collection empty — rebuilding from corpus (fallback)...")
    from src.chunker import build_chunk_records
    from src.vector_store import index_chunks
    with open("../data/raw/corpus.json") as _f:
        _corpus = json.load(_f)
    _chunks = build_chunk_records(_corpus, chunk_size=800, chunk_overlap=100)
    index_chunks(collection, _chunks, embed_model)
    print(f"Rebuilt: {collection.count()} docs indexed")

# --- LLM client ---
llm_client = OpenAI(
    base_url=os.getenv("NETLIGHT_BASE_URL", "https://llm.netlight.ai/v1"),
    api_key=os.getenv("NETLIGHT_API_KEY"),
)
FAST_MODEL = os.getenv("NETLIGHT_MODEL_FAST", "claude-haiku-4-5")
print(f"LLM: {FAST_MODEL}")

---
## Guardrail 1 — Input Scope Filter

Insurance chatbots regularly receive off-topic questions: general knowledge, medical advice,
jokes, prompt injections. A simple LLM-based classifier at the front door rejects them
before any retrieval happens — saving cost and preventing misuse.

**Your task:** implement `is_insurance_query(question)` that returns `True` only when
the question is about insurance, health coverage, or financial products.

**Hint:** keep the prompt short — ask for a single-word answer (`yes` / `no`).

In [ ]:
def is_insurance_query(question: str, client: OpenAI = llm_client, model: str = FAST_MODEL) -> bool:
    """Return True if the question is about insurance, health, or financial products."""
    # TODO: Call client.chat.completions.create with:
    #   system: brief classifier instruction
    #   user:   the question
    # Strip and lowercase the response, return True if it starts with "yes".
    raise NotImplementedError

In [ ]:
tests = [
    ("What is the annual deductible for DKV hospitalization?", True),
    ("What is the capital of Belgium?",                        False),
    ("Tell me a joke.",                                        False),
    ("Does my DKV plan cover physiotherapy?",                  True),
]

print(f"{'Question':<55} Expected  Got")
print("-" * 80)
for q, expected in tests:
    got = is_insurance_query(q)
    mark = "✔" if got == expected else "✘"
    print(f"{q:<55} {str(expected):<9} {got}  {mark}")

---
## Guardrail 2 — Retrieval Confidence Check

Even for in-scope questions the vector store may return poor matches — for example when
the question references a product or clause not present in the corpus.
If all retrieved chunks are low-similarity, the answer will likely be hallucinated.

**Your task:** implement `check_confidence(chunks, threshold)` that looks at the
`similarity` score of each retrieved chunk and decides whether retrieval was confident enough.

A threshold of **0.35** is a reasonable starting point for `all-MiniLM-L6-v2` cosine similarity.

In [ ]:
def check_confidence(chunks: list[dict], threshold: float = 0.35) -> tuple[bool, float]:
    """
    Return (passed, best_score).
    passed=True when at least one chunk exceeds the similarity threshold.
    """
    # TODO: Extract the 'similarity' value from each chunk dict.
    # Compute best_score = max of those values (or 0.0 if chunks is empty).
    # Return (best_score >= threshold, best_score).
    raise NotImplementedError

In [ ]:
questions = [
    "What does DKV cover for hospitalization?",  # in corpus — should pass
    "Does my plan cover cosmetic surgery?",      # may be borderline
    "Explain quantum entanglement.",             # off-corpus — should fail
]

for q in questions:
    chunks = retrieve(collection, q, top_k=5, model_name=EMBED_MODEL_NAME)
    passed, score = check_confidence(chunks)
    status = "✔ PASS" if passed else "✘ BLOCK"
    print(f"{status}  best={score:.3f}  '{q[:55]}'")

---
## Guardrail 3 — Faithfulness Gate

The first two guardrails are pre-generation checks.
This one is a **post-generation** check: given the answer the LLM produced,
is every factual claim actually supported by the retrieved context?

This is the same concept as the RAGAS `faithfulness` metric from Notebook 05,
but implemented as a lightweight inline check rather than an offline evaluation.

**Your task:** implement `check_faithfulness(answer, contexts)` that asks the LLM
to verify grounding and returns `(faithful: bool, reason: str)`.

**Hint:** ask for `yes` or `no` on the first line, followed by a one-sentence reason.

In [ ]:
def check_faithfulness(
    answer: str,
    contexts: list[str],
    client: OpenAI = llm_client,
    model: str = FAST_MODEL,
) -> tuple[bool, str]:
    """
    Ask the LLM whether the answer is supported by the context.
    Returns (faithful: bool, reason: str).
    """
    # TODO: Build a prompt that:
    #   - Presents the numbered context chunks
    #   - Presents the answer
    #   - Asks: is every factual claim in the answer supported by the context?
    #   Format: first line = yes/no, second line = one-sentence reason.
    # Parse the response — return (True, reason) if yes, else (False, reason).
    raise NotImplementedError

In [ ]:
test_question = "What physiotherapy sessions does DKV reimburse per year?"
chunks = retrieve(collection, test_question, top_k=5, model_name=EMBED_MODEL_NAME)
contexts = [c["text"] for c in chunks]

# Grounded answer — should be faithful
grounded_answer = llm_client.chat.completions.create(
    model=FAST_MODEL,
    max_tokens=256,
    messages=[
        {"role": "system", "content": "Answer using ONLY the context provided."},
        {"role": "user", "content": "Context:
" + "
".join(contexts) + "

Q: " + test_question},
    ],
).choices[0].message.content

# Hallucinated answer — should fail
hallucinated_answer = "DKV reimburses up to 60 physiotherapy sessions per year and covers 100% of costs."

for label, ans in [("Grounded", grounded_answer), ("Hallucinated", hallucinated_answer)]:
    faithful, reason = check_faithfulness(ans, contexts)
    mark = "✔" if faithful else "✘"
    print(f"{mark} {label}: {reason}")

---
## Exercise — `GuardedRAGPipeline`

Now combine all three guardrails into a single pipeline class.

The `ask` method should:

1. **Scope check** — if `is_insurance_query` returns `False`, return early with a refusal message
2. **Retrieve** top-5 chunks
3. **Confidence check** — if `check_confidence` fails, return early with a “no info found” message
4. **Generate** an answer using the LLM
5. **Faithfulness gate** — call `check_faithfulness`; set `flagged=True` if not faithful
6. **Return** a result dict with keys `question`, `answer`, and `guardrails`

```python
guardrails = {
    "scope_passed":  bool,
    "confidence":    float | None,  # best similarity score, or None if scope failed
    "faithfulness":  bool | None,   # True/False, or None if blocked earlier
    "flagged":       bool,           # True if any gate blocked or flagged
}
```

In [ ]:
class GuardedRAGPipeline:
    """
    Wraps retrieval + generation with three guardrails:
      1. is_insurance_query   — blocks off-topic questions
      2. check_confidence     — blocks low-similarity retrievals
      3. check_faithfulness   — flags answers not grounded in context
    """

    SYSTEM_PROMPT = (
        "You are a knowledgeable insurance assistant. "
        "Answer ONLY using information from the numbered context chunks provided. "
        "Cite sources using the chunk number, e.g. 'According to [1]...'. "
        "If the context is insufficient, say so explicitly. Never fabricate facts."
    )

    def __init__(
        self,
        collection,
        llm_client: OpenAI,
        confidence_threshold: float = 0.35,
        fast_model: str = FAST_MODEL,
    ):
        self.collection = collection
        self.llm_client = llm_client
        self.confidence_threshold = confidence_threshold
        self.fast_model = fast_model

    def ask(self, question: str) -> dict:
        """
        Run the guarded pipeline and return:
          {question, answer, guardrails: {scope_passed, confidence, faithfulness, flagged}}
        """
        # TODO: implement the 5-step flow described above.
        # For prompt building, iterate over chunks with enumerate(chunks, 1) and format:
        #   "[i] Source: {chunk['source']}\n{chunk['text']}\n"
        # Then append "\nQuestion: {question}\nAnswer:"
        raise NotImplementedError

In [ ]:
pipeline = GuardedRAGPipeline(collection, llm_client)

test_questions = [
    "What does DKV cover for dental care?",
    "Who wrote Hamlet?",
    "Does my policy cover medical repatriation abroad?",
]

for q in test_questions:
    result = pipeline.ask(q)
    g = result["guardrails"]
    flag = "⚠️ FLAGGED" if g["flagged"] else "✔ OK"
    print(f"\n{flag}  Q: {q}")
    print(f"  scope={g['scope_passed']}  conf={g['confidence']}  faithful={g['faithfulness']}")
    print(f"  A: {result['answer'][:120]}")

---
## Reflection

Think about the following questions:

1. **False positives in the scope filter** — a legitimate question like
   *“What is an excess clause?”* might be classified as out-of-scope.
   How would you reduce this error without letting irrelevant questions through?

2. **Confidence threshold tuning** — what happens if you set the threshold too high
   (e.g., 0.7)?  Too low (e.g., 0.1)?  How could you choose it systematically?

3. **Faithfulness latency** — the faithfulness gate adds a second LLM call to every
   request.  What are the cost/latency trade-offs?  When would you skip it?

4. **Flagging vs. blocking** — the faithfulness gate only *flags* the answer rather than
   blocking it.  In a real insurance product, should it block?  What would you show the user?

---
## Summary

| Guardrail | Where in pipeline | Cost |
|-----------|-------------------|------|
| Scope filter | Before retrieval | 1 LLM call |
| Confidence check | After retrieval | Free (threshold) |
| Faithfulness gate | After generation | 1 LLM call |

Together these three checks address the most common failure modes in production RAG:
off-topic queries, empty or weak retrievals, and hallucinated answers.
They are complementary to offline evaluation (RAGAS) — think of them as your real-time
safety net while RAGAS is your retrospective quality monitor.